In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [8]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}...")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
                doc.metadata["source_file"]=str(pdf_file)
                doc.metadata["file_type"]="pdf"
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file}")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")

    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents
all_pdf_documents =process_all_pdfs("../data")


found 1 PDF files to process.
Processing ..\data\pride-and-prejudice-jane-austen.pdf...
Loaded 515 pages from ..\data\pride-and-prejudice-jane-austen.pdf
Total documents loaded: 515


In [9]:
all_pdf_documents

[Document(metadata={'producer': '3-Heights™ PDF Optimization Shell 6.3.1.5 (http://www.pdf-tools.com)', 'creator': 'Adobe Acrobat Pro DC 20.9.20063', 'creationdate': '2022-07-01T09:23:19-04:00', 'author': 'jane austen', 'keywords': 'pride and prejudice by jane austen, pride, prejudice, jane austen, pride and prejudice, jane', 'moddate': '2022-07-01T14:27:51-04:00', 'title': 'pride and prejudice', 'source': '..\\data\\pride-and-prejudice-jane-austen.pdf', 'total_pages': 515, 'page': 0, 'page_label': '1', 'source_file': '..\\data\\pride-and-prejudice-jane-austen.pdf', 'file_type': 'pdf'}, page_content='PRIDE AND \nPREJUDICE\nJane Austen\nInfoBooks.org'),
 Document(metadata={'producer': '3-Heights™ PDF Optimization Shell 6.3.1.5 (http://www.pdf-tools.com)', 'creator': 'Adobe Acrobat Pro DC 20.9.20063', 'creationdate': '2022-07-01T09:23:19-04:00', 'author': 'jane austen', 'keywords': 'pride and prejudice by jane austen, pride, prejudice, jane austen, pride and prejudice, jane', 'moddate': 

### TEXT SPLITS INTO CHUNKS 

In [15]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_split=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n",".","!","?"," ", ""])
    split_doc = text_split.split_documents(documents)
    print(f" Split {len(documents)} documents into {len(split_doc)} chunks.")

    #Example of the first chunk
    if split_doc:
        print(f"\nExample of the first chunk:")
        print(f"Content: {split_doc[0].page_content[:500]}...")  # Print the first 500 characters   
        print(f"Metadata: {split_doc[0].metadata}")

    return split_doc

In [ ]:
chunks=split_documents(all_pdf_documents) 
chunks
print(f"Total chunks created: {len(chunks)}")

 Split 515 documents into 988 chunks.

Example of the first chunk:
Content: PRIDE AND 
PREJUDICE
Jane Austen
InfoBooks.org...
Metadata: {'producer': '3-Heights™ PDF Optimization Shell 6.3.1.5 (http://www.pdf-tools.com)', 'creator': 'Adobe Acrobat Pro DC 20.9.20063', 'creationdate': '2022-07-01T09:23:19-04:00', 'author': 'jane austen', 'keywords': 'pride and prejudice by jane austen, pride, prejudice, jane austen, pride and prejudice, jane', 'moddate': '2022-07-01T14:27:51-04:00', 'title': 'pride and prejudice', 'source': '..\\data\\pride-and-prejudice-jane-austen.pdf', 'total_pages': 515, 'page': 0, 'page_label': '1', 'source_file': '..\\data\\pride-and-prejudice-jane-austen.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': '3-Heights™ PDF Optimization Shell 6.3.1.5 (http://www.pdf-tools.com)', 'creator': 'Adobe Acrobat Pro DC 20.9.20063', 'creationdate': '2022-07-01T09:23:19-04:00', 'author': 'jane austen', 'keywords': 'pride and prejudice by jane austen, pride, prejudice, jane austen, pride and prejudice, jane', 'moddate': '2022-07-01T14:27:51-04:00', 'title': 'pride and prejudice', 'source': '..\\data\\pride-and-prejudice-jane-austen.pdf', 'total_pages': 515, 'page': 0, 'page_label': '1', 'source_file': '..\\data\\pride-and-prejudice-jane-austen.pdf', 'file_type': 'pdf'}, page_content='PRIDE AND \nPREJUDICE\nJane Austen\nInfoBooks.org'),
 Document(metadata={'producer': '3-Heights™ PDF Optimization Shell 6.3.1.5 (http://www.pdf-tools.com)', 'creator': 'Adobe Acrobat Pro DC 20.9.20063', 'creationdate': '2022-07-01T09:23:19-04:00', 'author': 'jane austen', 'keywords': 'pride and prejudice by jane austen, pride, prejudice, jane austen, pride and prejudice, jane', 'moddate': 

### EMBEDDING AND VECTOR STORE DB

In [17]:
import numpy as np
from sentence_transformers import SentenceTransformer # Embedding model
import chromadb
from typing import List, Dict, Any, Tuple
from chromadb.config import Settings
import uuid
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
class EmbeddingManager:
    """Handles embedding generation using SentenceTransformer and storage using ChromaDB"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager with the specified model 

        Args:
         model_name : Hugging Face model name for sentence embedding 
            """
        self.model_name = model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """load the sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding model: {self.model_name}, embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts

        Args:
            texts: List of strings to embed
        Returns:
            np.ndarray: Array of embeddings with shape (len(texts), embedding_dim)
            """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise

In [ ]:
embeddingmanager=EmbeddingManager()
embeddingmanager
print(f"Embedding model: {embeddingmanager.model_name}, embedding dimension: {embeddingmanager.model.get_embedding_dimension()}")

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3986.94it/s]


Loaded embedding model: all-MiniLM-L6-v2, embedding dimension: 384


### Vectorstore

In [ ]:
class vectorstore:
    """Manages document embeddings in a ChromaDB vector store   """
    def __init__(self,collection_name:str="pdf_document",persist_directory:str="../data/vector_store"):
        """initialize the vectorctore
            Args:
            collection_name: Name of the ChromaDB collection to use
            persist_directory: Directory where the ChromaDB data will be stored
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        """initialize the ChromaDB client and collection"""
        try:
            #create persist directory(chroma db client)
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            #get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"Collection of PDF document  chunks and embeddings"}
            )
            print(f"Initialized ChromaDB collection: {self.collection_name} at {self.persist_directory}")
            print(f"Current number of documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents:List[any],embeddings:np.ndarray):
        """ADD EMBEDDINGS AND DOCUMENTS TO THE VECTOR STORE
        Args:
            documents: List of Langchain document objects (with metadata)
            embeddings: np.ndarray of shape (len(documents), embedding_dim) containing the corresponding embeddings
        """
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings.")
        print(f"Adding {len(documents)} documents to the vector store...")

        #prepare data for ChromaDB
        ids =[]
        metadatas=[]
        document_texts=[]
        embeddings_list = []

        for i , (doc, embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_uuid_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata (
            metadata = dict(doc.metadata)  # copy existing metadatas
            metadata['doc_index'] =i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata) #git-version 
